# ShiftLog-Gym: GRPO Training Pipeline
## Meta PyTorch OpenEnv Hackathon Grand Finale 2026
**Runtime: T4 GPU (minimum) | Estimated time: 2–3 hours**

Stages:
- **A**: SFT format warmup (50 steps) — teaches JSON tool call format
- **B**: GRPO short rollout (200 steps) — learns recall-before-action
- **C**: GRPO full rollout (300 steps) — generalizes memory policy

After completion, runs evaluation and uploads everything to HuggingFace.

In [ ]:
import os

REPO_URL = "https://github.com/Chirag0096/ShiftLog-Gym.git"
REPO_DIR = "ShiftLog-Gym"

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    os.system("git pull")
else:
    os.system(f"git clone {REPO_URL}")
    os.chdir(REPO_DIR)

os.system("pip install -q -e . wandb matplotlib pandas seaborn huggingface_hub")
os.system("pip install -q trl==0.8.6 peft bitsandbytes accelerate")
# Pin trl to 0.8.6 — newer versions break GRPOTrainer environment_factory

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Switch runtime to T4/L4/A10G GPU before continuing.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
# T4 = 15GB, L4 = 24GB. Both work. A100 = 40GB (best).

In [ ]:
from getpass import getpass
import os

# Enter your keys when prompted
# OR set them as Colab secrets: Runtime → Manage secrets → add WANDB_API_KEY and HF_TOKEN
WANDB_KEY = os.environ.get("WANDB_API_KEY", "").strip()
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

if not WANDB_KEY:
    WANDB_KEY = getpass("Enter WANDB_API_KEY (from wandb.ai/settings): ").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter HF_TOKEN (from huggingface.co/settings/tokens): ").strip()

from train.colab_training_pipeline import ColabTrainingPipeline

pipe = ColabTrainingPipeline()
pipe.authenticate(wandb_key=WANDB_KEY, hf_token=HF_TOKEN)
print("WandB enabled:", pipe.wandb_enabled)
print("HuggingFace enabled:", pipe.hf_enabled)

In [ ]:
import wandb

run = wandb.init(
    project="shiftlog-gym",
    name="grpo-full-run-01",
    config={
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "stage_a_steps": 50,
        "stage_b_steps": 200,
        "stage_c_steps": 300,
        "lora_r": 8,
        "lora_alpha": 16,
        "dtype": "bf16",
        "environment": "https://huggingface.co/spaces/Chirag0123/shiftlog-gym",
        "hackathon": "Meta PyTorch OpenEnv 2026 Grand Finale",
    },
    tags=["shiftlog", "grpo", "sre", "memory", "hackathon"],
)
print("WandB run URL:", run.url)
print("WandB run ID:", run.id)
# SAVE THIS URL — you will put it in the README

In [ ]:
pipe.load_model("Qwen/Qwen2.5-1.5B-Instruct")
# This loads 4-bit quantized Qwen2.5-1.5B with LoRA (rank=8)
# BF16 compute dtype — no FP16 GradScaler (Qwen2.5 is a BFloat16 native model)
print("Model loaded successfully")
print("Trainable parameters above ↑")

In [ ]:
import requests

SPACE_URL = "https://chirag0123-shiftlog-gym.hf.space"

try:
    r = requests.post(f"{SPACE_URL}/reset", timeout=30)
    r.raise_for_status()
    obs = r.json()
    print("✅ Environment connected")
    print("Observation keys:", list(obs.keys()))
    r2 = requests.get(f"{SPACE_URL}/health", timeout=15)
    print("Health status:", r2.json().get("status", "unknown"))
except Exception as e:
    print(f"⚠️ Environment connection failed: {e}")
    print("Training will use local ShiftLogSimulator fallback — this is fine for Colab")

In [ ]:
print("=" * 60)
print("STAGE A — SFT Format Warmup (50 steps)")
print("Teaches the model to output strict JSON tool calls")
print("=" * 60)

pipe.run_stage_a(enabled=True)

# Log Stage A completion to WandB
if pipe.wandb_enabled:
    wandb.log({"stage_a_complete": 1})

print("✅ Stage A complete")
print("Artifacts:", list((pipe.outputs_dir / "stage-a-sft").glob("*")) if (pipe.outputs_dir / "stage-a-sft").exists() else "none saved")

In [ ]:
print("=" * 60)
print("STAGE B — GRPO Short Rollout (200 steps)")
print("Incident families: db_pool, auth_cascade, oom_regression")
print("Learning: recall-before-action on causally linked incidents")
print("=" * 60)

result_b = pipe.run_stage_grpo(pipe.stage_b)
print("Stage B result:", result_b)

if pipe.wandb_enabled:
    wandb.log({"stage_b_complete": 1, "stage_b_mode": result_b.get("mode", "unknown")})

# Show training curve
import pandas as pd
curve_b = pd.read_csv(pipe.runs_dir / "training_curves_stageB.csv")
print("\nStage B final metrics:")
print(curve_b.tail(3).to_string())

In [ ]:
print("=" * 60)
print("STAGE C — GRPO Full Rollout (300 steps)")
print("All incident families — generalizes memory policy")
print("Including noise resistance on independent incidents")
print("=" * 60)

result_c = pipe.run_stage_grpo(pipe.stage_c)
print("Stage C result:", result_c)

if pipe.wandb_enabled:
    wandb.log({"stage_c_complete": 1, "stage_c_mode": result_c.get("mode", "unknown")})

curve_c = pd.read_csv(pipe.runs_dir / "training_curves_stageC.csv")
print("\nStage C final metrics:")
print(curve_c.tail(5).to_string())

In [ ]:
print("=" * 60)
print("POST-TRAINING EVALUATION")
print("Running 20 episodes per stage — computing real MTTR numbers")
print("=" * 60)

summaries = pipe.evaluate_and_write()

print("\n=== EVALUATION RESULTS ===")
for stage, summary in summaries.items():
    print(f"\n{stage}:")
    for k, v in summary.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Read and display baselines.json
import json
baselines_path = pipe.obs_root / "baselines.json"
baselines = json.loads(baselines_path.read_text())
print("\n=== baselines.json (will be shown in Observatory dashboard) ===")
print(json.dumps(baselines, indent=2))

In [ ]:
pipe.generate_hackathon_plots()

# Display plots inline in Colab
from IPython.display import display, Image

for plot_name in ["01_reward_curve.png", "02_recall_bonus_curve.png", "03_mttr_comparison.png"]:
    plot_path = pipe.plots_dir / plot_name
    if plot_path.exists():
        print(f"\n{plot_name}:")
        display(Image(str(plot_path)))
    else:
        print(f"⚠️ {plot_name} not found — check generate_hackathon_plots()")

In [ ]:
print("=== ARTIFACT STATUS ===")
for path, status in pipe.artifact_status():
    icon = "✅" if status == "OK" else "❌"
    print(f"{icon} {status:8s} {path}")

# Also check plots
for plot in ["01_reward_curve.png", "02_recall_bonus_curve.png", "03_mttr_comparison.png"]:
    p = pipe.plots_dir / plot
    icon = "✅" if p.exists() and p.stat().st_size > 5000 else "❌"
    print(f"{icon} {'OK' if p.exists() else 'MISSING':8s} plots/{plot}")

In [ ]:
HF_MODEL_REPO = "Chirag0123/shiftlog-gym-qwen-memory-policy"

print(f"Uploading to: https://huggingface.co/{HF_MODEL_REPO}")

# Create model card before uploading
pipe.create_model_card()  # This method needs to be added (see BUG 6 fix)

pipe.upload_to_hf(HF_MODEL_REPO)

print(f"\n✅ Upload complete!")
print(f"Model: https://huggingface.co/{HF_MODEL_REPO}")
print(f"WandB: {wandb.run.url if wandb.run else 'not logged'}")

In [ ]:
# Download the 3 key plots so you can commit them to the Space repo
from google.colab import files

for plot in ["01_reward_curve.png", "02_recall_bonus_curve.png", "03_mttr_comparison.png"]:
    p = pipe.plots_dir / plot
    if p.exists():
        files.download(str(p))
        print(f"⬇ Downloaded: {plot}")
    else:
        print(f"⚠️ Missing: {plot}")

print("\n📋 After downloading, run these commands to commit to the Space repo:")
print("git clone https://huggingface.co/spaces/Chirag0123/shiftlog-gym")
print("cd shiftlog-gym")
print("cp ~/Downloads/0*.png plots/")
print('git add plots/*.png observatory/baselines.json')
print("git commit -m 'Add real GRPO training evidence: 3 PNG plots + real baselines'")
print("git push")

In [ ]:
import wandb as _wandb

print("=" * 60)
print("TRAINING COMPLETE — SUMMARY")
print("=" * 60)
print(f"WandB run:    {_wandb.run.url if _wandb.run else 'N/A'}")
print(f"Model repo:   https://huggingface.co/{HF_MODEL_REPO}")
print(f"Space:        https://huggingface.co/spaces/Chirag0123/shiftlog-gym")
print()
print("FILES TO COMMIT TO SPACE REPO:")
print("  plots/01_reward_curve.png")
print("  plots/02_recall_bonus_curve.png")
print("  plots/03_mttr_comparison.png")
print("  observatory/baselines.json   ← has real numbers now")
print()
print("BEFORE SUBMISSION DEADLINE:")
print("  1. Commit the 3 PNGs + baselines.json to the Space repo")
print("  2. Record the 2-minute demo video")
print("  3. Publish the HuggingFace blog post")
print("  4. Add the video + blog URL to README links section")

_wandb.finish()